# 📊 Mutual Fund Analytics
## Day 2: NAV History Data Cleaning

### 🎯 Objective
The objective of this notebook is to clean, validate, and prepare mutual fund NAV (Net Asset Value) history data for further analysis and SQL database integration.

### 📁 Dataset Information
- **Source:** `data/raw/02_nav_history.csv`
- **Output:** `data/processed/cleaned_nav_history.csv`
- **Records:** 46,000
- **Columns:** 3
- **Unique Mutual Fund Schemes:** 40

### 🛠️ Tools & Technologies
- Python
- Pandas
- Jupyter Notebook

### 🧹 Tasks Performed
1. Load and inspect the raw dataset.
2. Check missing values and duplicates.
3. Validate NAV values and AMFI scheme codes.
4. Convert dates into datetime format.
5. Sort records by scheme code and date.
6. Validate the cleaned dataset.
7. Export the processed data to CSV.

### ✅ Expected Outcome
A validated and chronologically sorted NAV history dataset ready for further data analysis and database integration.

In [1]:
import pandas as pd

print("Pandas imported successfully!")

Pandas imported successfully!


In [3]:
from pathlib import Path

# Move one level up from the notebooks folder
project_path = Path.cwd().parent

# Define the raw NAV file path
nav_file = project_path / "data" / "raw" / "02_nav_history.csv"

print("Project folder:", project_path)
print("NAV file exists:", nav_file.exists())
print("NAV file path:", nav_file)

Project folder: d:\Mutual-Fund-Analytics
NAV file exists: True
NAV file path: d:\Mutual-Fund-Analytics\data\raw\02_nav_history.csv


In [4]:
# Load the raw NAV history dataset
nav_df = pd.read_csv(nav_file)

print("NAV dataset loaded successfully!")
print("Dataset shape:", nav_df.shape)

NAV dataset loaded successfully!
Dataset shape: (46000, 3)


In [5]:
# Display the first five rows
display(nav_df.head())

# Check column names
print("\nColumn names:")
print(nav_df.columns.tolist())

# Check data types
print("\nData types:")
print(nav_df.dtypes)

# Check missing values
print("\nMissing values:")
print(nav_df.isnull().sum())

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692



Column names:
['amfi_code', 'date', 'nav']

Data types:
amfi_code      int64
date             str
nav          float64
dtype: object

Missing values:
amfi_code    0
date         0
nav          0
dtype: int64


In [6]:
# 1. Check duplicate rows
print("Duplicate rows:", nav_df.duplicated().sum())

# 2. Check duplicate fund-date combinations
print(
    "Duplicate fund-date combinations:",
    nav_df.duplicated(subset=["amfi_code", "date"]).sum()
)

# 3. Check NAV values that are zero or negative
print("NAV values <= 0:", (nav_df["nav"] <= 0).sum())

# 4. Check missing or invalid scheme codes
print("Invalid scheme codes:", (nav_df["amfi_code"] <= 0).sum())

# 5. Display a few records
display(nav_df.head())

Duplicate rows: 0
Duplicate fund-date combinations: 0
NAV values <= 0: 0
Invalid scheme codes: 0


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [7]:
# Create a separate DataFrame so the raw data remains unchanged
nav_clean = nav_df.copy()

# Convert date column from text to datetime
nav_clean["date"] = pd.to_datetime(
    nav_clean["date"],
    errors="coerce"
)

# Check whether any dates failed to convert
print("Invalid dates:", nav_clean["date"].isna().sum())

# Sort by scheme code and date
nav_clean = nav_clean.sort_values(
    by=["amfi_code", "date"]
).reset_index(drop=True)

# Display the cleaned data types and first five rows
print("\nUpdated data types:")
print(nav_clean.dtypes)

display(nav_clean.head())

Invalid dates: 0

Updated data types:
amfi_code             int64
date         datetime64[us]
nav                 float64
dtype: object


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [8]:
# 1. Check the overall date range
print("Earliest date:", nav_clean["date"].min())
print("Latest date:", nav_clean["date"].max())

# 2. Count unique funds
print("Unique funds:", nav_clean["amfi_code"].nunique())

# 3. Verify chronological sorting within each fund
is_sorted = (
    nav_clean.groupby("amfi_code")["date"]
    .apply(lambda dates: dates.is_monotonic_increasing)
    .all()
)

print("Chronologically sorted:", is_sorted)

# 4. Check whether any NAV values are missing
print("Missing NAV values:", nav_clean["nav"].isna().sum())

# 5. Confirm the final shape
print("Final dataset shape:", nav_clean.shape)

Earliest date: 2022-01-03 00:00:00
Latest date: 2026-05-29 00:00:00
Unique funds: 40
Chronologically sorted: True
Missing NAV values: 0
Final dataset shape: (46000, 3)


In [9]:
# Define the output folder
processed_path = project_path / "data" / "processed"

# Create the folder if it doesn't exist
processed_path.mkdir(parents=True, exist_ok=True)

# Define the output file
output_file = processed_path / "cleaned_nav_history.csv"

# Export the cleaned DataFrame
nav_clean.to_csv(output_file, index=False)

print("NAV history saved successfully!")
print("Output file:", output_file)
print("Rows exported:", len(nav_clean))

NAV history saved successfully!
Output file: d:\Mutual-Fund-Analytics\data\processed\cleaned_nav_history.csv
Rows exported: 46000


# Mutual Fund Analytics — NAV History Data Cleaning

## 1. Project Overview
This notebook focuses on cleaning and validating the historical Net Asset Value (NAV) data used in the Mutual Fund Analytics project.

The objective is to improve data quality, standardize data types, verify data integrity, and prepare the dataset for SQL-based analytics.

## 2. Dataset Information
- **Source file:** `data/raw/02_nav_history.csv`
- **Output file:** `data/processed/cleaned_nav_history.csv`
- **Total records:** 46,000
- **Total columns:** 3
- **Unique mutual fund schemes:** 40

### Columns
| Column | Description | Data Type |
|---|---|---|
| `amfi_code` | AMFI mutual fund scheme code | Integer |
| `date` | NAV record date | Datetime |
| `nav` | Net Asset Value of the scheme | Float |

## 3. Data Cleaning Steps
1. Loaded the raw NAV history CSV using Pandas.
2. Inspected the dataset structure, column names, and data types.
3. Checked for missing values.
4. Checked for duplicate rows.
5. Checked for duplicate combinations of `amfi_code` and `date`.
6. Validated that NAV values are greater than zero.
7. Checked that AMFI scheme codes are positive.
8. Converted the `date` column to datetime format.
9. Sorted the records by `amfi_code` and `date`.
10. Validated chronological ordering and exported the processed dataset.

## 4. Data Quality Results
| Validation Check | Result |
|---|---:|
| Total records | 46,000 |
| Missing values | 0 |
| Duplicate rows | 0 |
| Duplicate fund-date combinations | 0 |
| NAV values less than or equal to zero | 0 |
| Invalid dates | 0 |
| Non-positive scheme codes | 0 |
| Unique schemes | 40 |

## 5. Dataset Date Range
- **Start date:** 2022-01-03
- **End date:** 2026-05-29

## 6. Conclusion
The NAV history dataset passed the data-quality checks performed in this notebook.

The date column was converted to datetime format, and records were sorted chronologically within each mutual fund scheme.

The processed dataset was exported to CSV for subsequent analysis and database loading.

**Note:** Weekend and holiday NAV handling must be reviewed against the project requirements before the NAV cleaning task is considered fully complete.